# Hyperparameter Tuning — CINIC-10 CNN

This notebook runs the hyperparameter sweep and visualizes results.

In [ ]:
import os, sys
sys.path.insert(0, os.path.join('..', 'src'))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils import get_device, set_seeds

set_seeds(42)
device = get_device()

DATA_DIR  = os.path.join('..', 'data')
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
VAL_DIR   = os.path.join(DATA_DIR, 'valid')
EPOCHS = 3  # increase for final submission
BATCH_SIZE = 32
print(f'TRAIN_DIR: {TRAIN_DIR}')
print(f'VAL_DIR: {VAL_DIR}')
print(f'Using device: {device}')
print('Setup complete')

## 1. Learning Rate Analysis

In [ ]:
from model_architecture import create_baseline_cnn
from hyperparameter_analysis import analyze_learning_rates

lr_results = analyze_learning_rates(
    create_baseline_cnn, TRAIN_DIR, VAL_DIR,
    learning_rates=[0.0001, 0.001, 0.01, 0.1],
    epochs=EPOCHS
)
lr_df = pd.DataFrame(lr_results)
print(lr_df[['learning_rate', 'val_accuracy', 'train_accuracy']])

## 2. Batch Size Analysis

In [ ]:
from hyperparameter_analysis import analyze_batch_sizes

batch_results = analyze_batch_sizes(
    create_baseline_cnn, TRAIN_DIR, VAL_DIR,
    batch_sizes=[16, 32, 64, 128],
    epochs=EPOCHS
)
batch_df = pd.DataFrame(batch_results)
print(batch_df[['batch_size', 'val_accuracy', 'train_accuracy']])

## 3. Regularization Analysis

In [ ]:
from model_architecture import create_cnn_with_regularization
from hyperparameter_analysis import analyze_regularization_strengths

reg_results = analyze_regularization_strengths(
    create_cnn_with_regularization, TRAIN_DIR, VAL_DIR,
    dropout_rates=[0.1, 0.3, 0.5],
    weight_decays=[1e-4, 1e-3],
    epochs=EPOCHS
)
reg_df = pd.DataFrame(reg_results)
print(reg_df[['dropout_rate', 'weight_decay', 'val_accuracy']])

## 4. Optimizer Comparison

In [ ]:
from hyperparameter_analysis import analyze_optimizers

opt_results = analyze_optimizers(
    create_baseline_cnn, TRAIN_DIR, VAL_DIR,
    optimizers=['adam', 'sgd', 'rmsprop'],
    epochs=EPOCHS
)
opt_df = pd.DataFrame(opt_results)
print(opt_df[['optimizer', 'val_accuracy', 'train_accuracy']])

## 5. Visualizations

In [ ]:
from hyperparameter_analysis import plot_hyperparameter_results

plot_hyperparameter_results(lr_df, title='Learning Rate Analysis')

## Summary

- **Learning rate**: lower LRs (1e-4, 1e-3) typically converge more stably
- **Batch size**: smaller batches provide more gradient updates per epoch
- **Regularization**: higher dropout reduces overfitting
- **Optimizer**: Adam generally outperforms SGD for CNNs